# Potential-field matched bandpass lab
Separate a synthetic grid into two equivalent-layer components using a supplied radial spectral model. This is a bounded amplitude-filter baseline; it does not fit source layers automatically.

In [ ]:
import sys
sys.path.insert(0, '../implementations/python/potential-field')
import numpy as np
import matplotlib.pyplot as plt
from matched_filtering import matched_bandpass_decomposition
n = 128
y, x = np.mgrid[:n, :n]
shallow = np.cos(2*np.pi*x/8) + .5*np.sin(2*np.pi*y/12)
deep = 1.5*np.cos(2*np.pi*x/48) + .8*np.sin(2*np.pi*y/40)
field = shallow + deep + 3.0
layers = [{'B': 1.0, 'z': 2.0, 'n': 0}, {'B': .8, 'z': 15.0, 'n': 0}]
result = matched_bandpass_decomposition(field, 1, 1, layers, dc_layer=0)
components = result['components']
print('Reconstruction RMSE:', np.sqrt(np.mean((components.sum(axis=0)-field)**2)))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, image, label in zip(axes, [field, components[0], components[1]], ['Input field','Equivalent layer 1','Equivalent layer 2']):
    view = ax.imshow(image, origin='lower'); ax.set_title(label); fig.colorbar(view, ax=ax)
plt.show()
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(result['filters'][0, 0, :n//2], label='Layer 1')
ax.plot(result['filters'][1, 0, :n//2], label='Layer 2')
ax.set_xlabel('Horizontal Fourier index'); ax.set_ylabel('Amplitude weight'); ax.legend(); plt.show()

## Interpretation limits
The component separation depends on the supplied equivalent-layer amplitudes, depths and spectral exponents. The FFT assumes a regular periodic grid; edge discontinuities can cause leakage. The DC mean is assigned to the selected layer as an explicit bookkeeping convention. Inspect radial spectra and model fit before geological interpretation.